In [0]:
"""
id: source_0
template: source
templateVersion: 3.0.0
name: sales
position:
  x: 65.39996337890625
  y: 172.9000015258789
description:
  text: Read data from a CSV file located at a specified path.
  hash: 5178273a
codegenHash: d4320a4d29458ff7
updatedAt: 1794903398288
config:
  file_source:
    path: /Volumes/dev/bronze/raw/sales/sales.csv
    format: '"csv"'
    schemaEvolutionMode: '"none"'
    ignoreLeadingWhiteSpace: false
    ignoreTrailingWhiteSpace: false
    inferColumnTypes: false
    dateFormat: '""'
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

from pyspark.errors import PySparkException

def _lb_read_files(file_source, spark):
    args = {"_lb_source_path": file_source["path"]}
    options = []
    for key, value in file_source.items():
        if key == "path":
            continue
        quoted_key = chr(96) + key.replace(chr(96), chr(96) * 2) + chr(96)
        name = "_lb_source_option_" + str(len(options))
        args[name] = value
        options.append(f"{quoted_key} => :{name}")
    arguments = ", ".join([":_lb_source_path", *options])
    return spark.sql(f"SELECT * FROM read_files({arguments})", args=args)


def _lb_decode_file_string(value):
    escapes = {"0": chr(0), "b": chr(8), "n": chr(10), "r": chr(13), "t": chr(9), "Z": chr(26)}
    decoded = []
    index = 0
    while index < len(value):
        if value[index] != chr(92) or index + 1 == len(value):
            decoded.append(value[index])
            index += 1
            continue
        index += 1
        char = value[index]
        width = 4 if char == "u" else 8 if char == "U" else 0
        digits = value[index + 1:index + 1 + width]
        octal = value[index:index + 3]
        if width and len(digits) == width and all(c in "0123456789abcdefABCDEF" for c in digits):
            codepoint = int(digits, 16)
            if codepoint >= 0x10000:
                high_surrogate = (((codepoint - 0x10000) >> 10) + 0xD800) & 0xFFFF
                decoded.extend((chr(high_surrogate), chr((codepoint & 0x3FF) + 0xDC00)))
            else:
                decoded.append(chr(codepoint))
            index += width + 1
        elif char in "01" and len(octal) == 3 and all(c in "01234567" for c in octal):
            decoded.append(chr(int(octal, 8)))
            index += 3
        else:
            decoded.append(chr(92) + char if char in "%_" else escapes.get(char, char))
            index += 1
    utf16 = "".join(decoded).encode("utf-16", "surrogatepass").decode("utf-16", "surrogatepass")
    return utf16.encode("utf-8", "replace").decode("utf-8")

def _lb_read_source_files(file_source, spark):
    path = file_source.get("path")
    if not path:
        raise ValueError("Source: 'path' is required for file source")
    try:
        escaped_literals = spark.conf.get("spark.sql.parser.escapedStringLiterals", "false") == "true"
    except PySparkException as error:
        condition_getter = getattr(error, "getCondition", None)
        condition = condition_getter() if callable(condition_getter) else None
        if not condition:
            legacy_getter = getattr(error, "getErrorClass", None)
            condition = legacy_getter() if callable(legacy_getter) else None
        if (condition or "").split(".")[0] != "CONFIG_NOT_AVAILABLE":
            raise
        escaped_literals = False
    if not escaped_literals:
        path = _lb_decode_file_string(path)
    normalized = {"path": path}
    for key, value in file_source.items():
        if key == "path":
            continue
        if key == "headerRows" and isinstance(value, bool):
            value = int(value)
        elif not isinstance(value, (bool, int, float)):
            value = str(value)
            if len(value) >= 2 and value[0] in ("'", '"') and value[-1] == value[0]:
                value = value[1:-1]
            if key == "dataAddress" and value.startswith("!"):
                value = value[1:]
            if not escaped_literals:
                value = _lb_decode_file_string(value)
        if key.startswith(chr(96)) and key.endswith(chr(96)):
            key = key[1:-1].replace(chr(96) * 2, chr(96))
        normalized[key] = value
    return _lb_read_files(normalized, spark)


def _build_metric_view_sql(
    dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        raise ValueError("Source: select at least one metric view dimension or measure")
    sql = f"SELECT {', '.join(select_parts)} FROM IDENTIFIER(:_lb_source_table)"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM IDENTIFIER(:_lb_source_table)"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(
    payload: Dict[str, Any], file_url: str, spreadsheet_range: Optional[str]
) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = None if spreadsheet_range else _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        for sheet in content.values():
            error = sheet.get("error") if isinstance(sheet, dict) else None
            if isinstance(error, str) and error:
                raise ValueError(f"Managed MCP spreadsheet read failed: {error}")
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(
    result: Dict[str, Any], file_url: str, spreadsheet_range: Optional[str], spark
):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url, spreadsheet_range)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        out = _lb_read_source_files(file_source, spark)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(list(dims), list(measures))
            out = spark.sql(sql, args={"_lb_source_table": table_name})
        elif table_source.get("isExpression") is True:
            param_names = set(re.findall(r"(?<!:):(\w+)", table_name))
            widgets = {}
            if param_names and "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
                widgets = dict(dbutils.widgets.getAll())
            args = {name: widgets[name] for name in param_names if name in widgets}
            out = spark.sql(table_name, args=args) if args else spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(
            result,
            mcp_file_source.get("fileUrl", ""),
            mcp_file_source.get("spreadsheetRange"),
            spark,
        )

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/dev/bronze/raw/sales/sales.csv",
        "format": "\"csv\"",
        "schemaEvolutionMode": "\"none\"",
        "ignoreLeadingWhiteSpace": False,
        "ignoreTrailingWhiteSpace": False,
        "inferColumnTypes": False,
        "dateFormat": "\"\""
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_0.data"])
if "source_0" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_0", "counts": {"data": int(ctx["source_0.data"].count())}}}, raw=True)

In [0]:
"""
id: filter_1
template: filter
templateVersion: 3.0.0
name: filter_1
position:
  x: 302.39996337890625
  y: 211.9000015258789
description:
  text: Keep rows where all specified columns are not empty; separate data into matching and non-matching sets.
  hash: 0b194483
codegenHash: 0b765c6e217e4035
updatedAt: 1794903567759
config:
  condition: order_id IS NOT NULL AND customer_id IS NOT NULL AND transaction_id IS NOT NULL AND product_id IS NOT NULL AND quantity IS NOT NULL AND discount_amount IS NOT NULL AND total_amount IS NOT NULL AND order_date IS NOT NULL
  condition_model:
    operator: AND
    groups:
      - id: g_rGIJ0kTK
        operator: OR
        conditions:
          - id: c_LjUJvgjM
            type: structured
            expression: order_id IS NOT NULL
      - id: g_rhdm8BYT
        operator: OR
        conditions:
          - id: c_HPoMCaTY
            type: structured
            expression: customer_id IS NOT NULL
      - id: g_5ucAlemw
        operator: OR
        conditions:
          - id: c_bec8BNWi
            type: structured
            expression: transaction_id IS NOT NULL
      - id: g_XCVisEIV
        operator: OR
        conditions:
          - id: c_lUlISAYa
            type: structured
            expression: product_id IS NOT NULL
      - id: g_Zy29sq28
        operator: OR
        conditions:
          - id: c_eiy2aQTw
            type: structured
            expression: quantity IS NOT NULL
      - id: g__JqrEolr
        operator: OR
        conditions:
          - id: c_0DdMsmRo
            type: structured
            expression: discount_amount IS NOT NULL
      - id: g_6-jceQhv
        operator: OR
        conditions:
          - id: c_0LaKj5B5
            type: structured
            expression: total_amount IS NOT NULL
      - id: g_Yj8w8NJM
        operator: OR
        conditions:
          - id: c_QLPcSiOW
            type: structured
            expression: order_date IS NOT NULL
checkpoint:
  mode: auto
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Dict, Any
from uuid import uuid4

from pyspark.sql import functions as F

_LIST_PARAMETER_TAIL = re.compile(
    r"\bIN\s*\(\s*:(`(?:[^`]|``)+`|[A-Za-z_][A-Za-z0-9_]*)\s*\)\s*$",
    re.IGNORECASE,
)

def _expand_list_parameter(expression, name, args, expanded_args):
    """Binds each comma-separated item of the trailing `IN (:name)` as its own marker."""
    match = _LIST_PARAMETER_TAIL.search(expression)
    if match is None or name not in args:
        return expression
    token = match.group(1)
    if (token[1:-1].replace("``", "`") if token.startswith("`") else token) != name:
        return expression
    items = [item.strip() for item in str(args[name]).split(",")]
    items = [item for item in items if item]
    if not items:
        raise ValueError("Parameter :" + name + " has no values.")
    markers = []
    for item in items:
        key = "_lb_list_" + str(len(expanded_args) - len(args))
        expanded_args[key] = item
        markers.append(":" + key)
    return expression[: match.start()] + "IN (" + ", ".join(markers) + ")"

def _combine(conditions, operator):
    combined = conditions[0]
    for condition in conditions[1:]:
        combined = combined & condition if operator == "AND" else combined | condition
    return combined

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    legacy_condition = config.get("condition", "")
    condition_model = config.get("condition_model")
    condition_model = condition_model or {}
    if config.get("__lb_bind_filter_conditions"):
        args = dict(dbutils.widgets.getAll())
        expanded_args = dict(args)
        sql_groups = []
        for group in condition_model.get("groups", []):
            expressions = []
            for unit in group.get("conditions", []):
                expression = unit.get("expression", "")
                if not expression.strip():
                    continue
                if unit.get("__lb_list_parameter"):
                    expression = _expand_list_parameter(
                        expression, unit["__lb_list_parameter"], args, expanded_args
                    )
                expressions.append("(" + expression + "\n)")
            if expressions:
                group_operator = " AND " if group.get("operator") == "AND" else " OR "
                sql_groups.append("(" + group_operator.join(expressions) + ")")
        if sql_groups:
            model_operator = " AND " if condition_model.get("operator") == "AND" else " OR "
            predicate = model_operator.join(sql_groups)
            view_name = "_lb_filter_input_" + uuid4().hex
            keep_name = "_lb_filter_keep_" + uuid4().hex
            while keep_name in df.columns:
                keep_name = "_lb_filter_keep_" + uuid4().hex
            df.createOrReplaceTempView(view_name)
            try:
                query = (
                    "SELECT *, COALESCE((" + predicate + "\n), false) AS `" + keep_name + "` "
                    "FROM `" + view_name + "`"
                )
                evaluated = spark.sql(query, args=expanded_args)
            except Exception:
                spark.catalog.dropTempView(view_name)
                raise
            globals().setdefault("_lb_views", set()).add(view_name)
            keep = F.col(keep_name)
            return {
                "filtered_data": evaluated.filter(keep).drop(keep_name),
                "excluded_data": evaluated.filter(~keep).drop(keep_name),
            }
    group_results = []

    for group in condition_model.get("groups", []):
        conditions = [
            F.expr(unit["expression"])
            for unit in group.get("conditions", [])
            if unit.get("expression", "").strip()
        ]
        if conditions:
            group_results.append(_combine(conditions, group.get("operator", "OR")))

    if group_results:
        final_condition = _combine(group_results, condition_model.get("operator", "AND"))
    elif isinstance(legacy_condition, str) and legacy_condition.strip():
        final_condition = F.expr(legacy_condition)
    else:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    return {
        "filtered_data": df.filter(final_condition),
        "excluded_data": df.filter(~F.coalesce(final_condition, F.lit(False))),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "order_id IS NOT NULL AND customer_id IS NOT NULL AND transaction_id IS NOT NULL AND product_id IS NOT NULL AND quantity IS NOT NULL AND discount_amount IS NOT NULL AND total_amount IS NOT NULL AND order_date IS NOT NULL",
    "condition_model": {
        "operator": "AND",
        "groups": [
            {
                "id": "g_rGIJ0kTK",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_LjUJvgjM",
                        "type": "structured",
                        "expression": "order_id IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_rhdm8BYT",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_HPoMCaTY",
                        "type": "structured",
                        "expression": "customer_id IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_5ucAlemw",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_bec8BNWi",
                        "type": "structured",
                        "expression": "transaction_id IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_XCVisEIV",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_lUlISAYa",
                        "type": "structured",
                        "expression": "product_id IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_Zy29sq28",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_eiy2aQTw",
                        "type": "structured",
                        "expression": "quantity IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g__JqrEolr",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_0DdMsmRo",
                        "type": "structured",
                        "expression": "discount_amount IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_6-jceQhv",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_0LaKj5B5",
                        "type": "structured",
                        "expression": "total_amount IS NOT NULL"
                    }
                ]
            },
            {
                "id": "g_Yj8w8NJM",
                "operator": "OR",
                "conditions": [
                    {
                        "id": "c_QLPcSiOW",
                        "type": "structured",
                        "expression": "order_date IS NOT NULL"
                    }
                ]
            }
        ]
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_0.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_1.filtered_data"] = out["filtered_data"]
ctx["filter_1.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_1.filtered_data"])
    display(ctx["filter_1.excluded_data"])
if "filter_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_1", "counts": {"filtered_data": int(ctx["filter_1.filtered_data"].count()), "excluded_data": int(ctx["filter_1.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: prepare_7b4c149a
template: prepare
templateVersion: 3.0.0
name: Fix data types
position:
  x: 571.0844093624376
  y: 218.4703428073552
description:
  text: Convert quantity to integer, discount and total amounts to double, and order date to date format, replacing errors with nulls.
  hash: a05b3c59
previewCodeHash: 20312d29374e8601
codegenHash: e39d4dc54f999a7f
updatedAt: 1794903927235
config:
  actions:
    - type: cast
      id: a_KAFhpCSu
      enabled: true
      column: quantity
      to: int
      on_error: "null"
    - type: cast
      id: a_KN8k4BI9
      enabled: true
      column: discount_amount
      to: double
      on_error: "null"
    - type: cast
      id: a_-uP5achX
      enabled: true
      column: total_amount
      to: double
      on_error: "null"
    - type: cast
      id: a_oQ-YmrBO
      enabled: true
      column: order_date
      to: date
      on_error: "null"
checkpoint:
  mode: auto
input:
  - node: filter_1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from functools import partial
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
from pyspark.errors import AnalysisException, ParseException
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())


def _lb_sql_string_literal(value):
    encoded = str(value).encode("utf-8").hex()
    return f"decode(X'{encoded}', 'UTF-8')"

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    return F.expr(f"try_cast({_lb_sql_string_literal(value)} as {target_dtype})")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _formula_references(expression):
    if not isinstance(expression, str) or not expression.isascii() or len(expression) > 4096:
        return None
    if chr(92) in expression or "--" in expression or "/*" in expression:
        return None
    quote = chr(96)
    token_pattern = (
        "[ \t\r\n]+|'(?:[^']|'')*'|" + quote + "(?:[^" + quote + "]|" + quote * 2 + ")*" + quote
        + "|[0-9]+(?:[.][0-9]+)?(?:[eE][+-]?[0-9]+)?|[A-Za-z_][A-Za-z_0-9]*"
        + "|<=|>=|<>|!=|==|[()+*/%=<>,-]"
    )
    tokens = []
    position = 0
    while position < len(expression):
        match = re.match(token_pattern, expression[position:])
        if match is None:
            return None
        token = match.group(0)
        position += len(token)
        if not token.isspace():
            tokens.append(token)
        if len(tokens) > 512:
            return None
    if any(token.upper() in {"CURRENT_TIME", "CURRENT_DATE", "CURRENT_TIMESTAMP", "CURRENT_USER", "SESSION_USER", "USER", "LOCALTIMESTAMP", "CURRENT_CATALOG", "CURRENT_SCHEMA"} for token in tokens):
        return None
    if not tokens:
        return None
    index = 0
    references = set()
    keywords = {"AND", "OR", "NOT", "IS", "NULL", "TRUE", "FALSE", "CASE", "WHEN", "THEN", "ELSE", "END"}

    def take(value):
        nonlocal index
        if index < len(tokens) and tokens[index].upper() == value:
            index += 1
            return True
        return False

    def require(value):
        if not take(value):
            raise ValueError()

    def atom():
        nonlocal index
        if take("("):
            boolean()
            require(")")
        elif take("CASE"):
            if not take("WHEN"):
                boolean()
                require("WHEN")
            boolean()
            require("THEN")
            boolean()
            while take("WHEN"):
                boolean()
                require("THEN")
                boolean()
            if take("ELSE"):
                boolean()
            require("END")
        elif index < len(tokens):
            token = tokens[index]
            index += 1
            if token.startswith(quote):
                references.add(token[1:-1].replace(quote * 2, quote).lower())
            elif token[0].isdigit() or token.startswith("'") or token.upper() in {"NULL", "TRUE", "FALSE"}:
                pass
            elif re.fullmatch("[A-Za-z_][A-Za-z_0-9]*", token) and token.upper() not in keywords:
                references.add(token.lower())
            else:
                raise ValueError()
        else:
            raise ValueError()

    def unary():
        if take("+") or take("-"):
            unary()
        else:
            atom()

    def product():
        unary()
        while take("*") or take("/") or take("%"):
            unary()

    def addition():
        product()
        while take("+") or take("-"):
            product()

    def comparison():
        addition()
        if take("IS"):
            take("NOT")
            require("NULL")
        elif any(take(operator) for operator in ("=", "==", "!=", "<>", "<=", ">=", "<", ">")):
            addition()

    def negation():
        if take("NOT"):
            negation()
        else:
            comparison()

    def conjunction():
        negation()
        while take("AND"):
            negation()

    def boolean():
        conjunction()
        while take("OR"):
            conjunction()

    try:
        boolean()
        return references if index == len(tokens) else None
    except (ValueError, RecursionError):
        return None

def _flush_formulas(df, pending, spark):
    if len(pending) == 1:
        return _apply_formula(df, pending[0], spark)
    if pending:
        return df.withColumns({action["target"]: F.expr(action["expression"]) for action in pending})
    return df

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _validate_cast_type(spark, action: Dict[str, Any]) -> str:
    to_type = action.get("to", "")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    try:
        spark.range(0).select(F.lit(None).cast(to_type)).schema
    except (AnalysisException, ParseException) as exc:
        raise ValueError("cast: 'to' must be a valid Spark SQL data type") from exc
    return to_type

def _apply_cast(df, action: Dict[str, Any], column: str, *, validated_type: str):
    on_error = action.get("on_error", "null")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {validated_type})")
    return _col(column).cast(validated_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    if action_type == "cast":
        build_expression = partial(build_expression, validated_type=_validate_cast_type(df.sparkSession, action))
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def _split_list_parameter(value: Any, action_type: str) -> List[str]:
    items = [item.strip() for item in str(value or "").split(",")]
    items = [item for item in items if item]
    if not items:
        raise ValueError(f"{action_type}: the 'columns' parameter has no column names")
    return items

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    pending = []
    pending_reads = set()
    pending_writes = set()
    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            df = _flush_formulas(df, pending, spark)
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action.get("__lb_split_columns"):
            action = {**action, "columns": _split_list_parameter(action.get("columns"), action_type)}
        if action_type == "formula" and not action.get("__lb_bind_expression"):
            target = action.get("target")
            references = _formula_references(action.get("expression"))
            if isinstance(target, str) and target and target.isascii() and references is not None:
                write = target.lower()
                if write in pending_writes or write in pending_reads or references & pending_writes:
                    df = _flush_formulas(df, pending, spark)
                    pending, pending_reads, pending_writes = [], set(), set()
                pending.append(action)
                pending_reads.update(references)
                pending_writes.add(write)
                continue
        df = _flush_formulas(df, pending, spark)
        pending, pending_reads, pending_writes = [], set(), set()
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    df = _flush_formulas(df, pending, spark)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "cast",
            "id": "a_KAFhpCSu",
            "enabled": True,
            "column": "quantity",
            "to": "int",
            "on_error": "null"
        },
        {
            "type": "cast",
            "id": "a_KN8k4BI9",
            "enabled": True,
            "column": "discount_amount",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "id": "a_-uP5achX",
            "enabled": True,
            "column": "total_amount",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "id": "a_oQ-YmrBO",
            "enabled": True,
            "column": "order_date",
            "to": "date",
            "on_error": "null"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_1.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_7b4c149a.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_7b4c149a" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_7b4c149a.prepared_data"])
if "prepare_7b4c149a" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_7b4c149a", "counts": {"prepared_data": int(ctx["prepare_7b4c149a.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: unique_ffc27a21
template: unique
templateVersion: 1.0.0
name: Remove duplicates
position:
  x: 823.5710293175893
  y: 218.47034280735522
description:
  text: Keep only unique rows based on order_id column.
  hash: fca16615
previewCodeHash: 14c127d0fed27e44
codegenHash: f37e4cbd06314ad6
updatedAt: 1794919315631
config:
  unique_by_all_columns: false
  columns:
    - order_id
  sort_expressions: []
checkpoint:
  mode: auto
input:
  - node: prepare_7b4c149a
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F
from pyspark.sql import Window

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError("Unique operator requires input 'data'")
    unique_by_all_columns = config.get("unique_by_all_columns", True)
    columns = config.get("columns", [])
    if not isinstance(columns, list):
        columns = []
    else:
        columns = [c for c in columns if isinstance(c, str)]
    sort_expressions = config.get("sort_expressions", [])

    if unique_by_all_columns:
        columns = df.columns
    elif not columns:
        return {"unique_data": df}

    order_idx = "__lb_orig_order__"
    while order_idx in df.columns:
        order_idx = order_idx + "_"
    df = df.withColumn(order_idx, F.monotonically_increasing_id())

    if unique_by_all_columns or not sort_expressions:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    order_cols = []
    for sort_def in sort_expressions:
        if not isinstance(sort_def, dict):
            continue
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        if not raw_expr:
            continue
        direction = sort_def.get("sortBy", "ASC")
        col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        if direction == "DESC":
            col = col.desc_nulls_last()
        elif direction == "ASC":
            col = col.asc_nulls_last()
        order_cols.append(col)
    
    if not order_cols:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    rn = "__lb_row_number__"
    while rn in df.columns:
        rn = rn + "_"

    window = Window.partitionBy(*[F.col(c) for c in columns]).orderBy(*order_cols, F.col(order_idx))
    tagged = df.withColumn(rn, F.row_number().over(window))
    unique = tagged.filter(F.col(rn) == 1).orderBy(order_idx).drop(rn, order_idx)
    return {"unique_data": unique}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "unique_by_all_columns": False,
    "columns": [
        "order_id"
    ],
    "sort_expressions": []
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_7b4c149a.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["unique_ffc27a21.unique_data"] = out["unique_data"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["unique_ffc27a21.unique_data"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["unique_ffc27a21.unique_data"]
        try:
            ctx["unique_ffc27a21.unique_data"] = ctx["unique_ffc27a21.unique_data"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["unique_ffc27a21.unique_data"] is not _lb_cp_original_0:
                try:
                    ctx["unique_ffc27a21.unique_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["unique_ffc27a21.unique_data"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "unique_ffc27a21" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["unique_ffc27a21.unique_data"])
if "unique_ffc27a21" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "unique_ffc27a21", "counts": {"unique_data": int(ctx["unique_ffc27a21.unique_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_b89e1903
template: aggregate
templateVersion: 2.0.0
name: Total number of orders
position:
  x: 1143.175620232562
  y: 294.481759351854
description:
  text: Count total orders.
  hash: 8cb4b94e
previewCodeHash: c73431ed0af7ad9c
codegenHash: 39e07f1e5701539d
updatedAt: 1794915626469
config:
  group_bys: []
  aggregations:
    - columnExpr:
        expr: order_id
        type: expr
      fn: COUNT
      alias: total_orders
checkpoint:
  mode: auto
input:
  - node: unique_ffc27a21
    input_port: data
    output_port: unique_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def _quoted_column(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _column_name(column_expr: Dict[str, Any], df) -> str:
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "column" and name != "*" and name in df.columns:
        return _quoted_column(name)
    return name

def _column_arg(column_expr: Dict[str, Any], df):
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "expr":
        return F.expr(name)
    return F.col(_column_name(column_expr, df))

def _sql_arg(column_expr: Dict[str, Any], df) -> str:
    return _column_name(column_expr, df)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(_column_arg(col_expr, df))
        elif fn == "-" or fn == "_":
            col = _column_arg(col_expr, df)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({_sql_arg(col_expr, df)}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            col = F.concat_ws(sep, F.collect_list(_column_arg(col_expr, df)))
        elif fn == "COUNT_DISTINCT":
            arg = _column_arg(col_expr, df) if col_expr.get("type") == "column" else F.expr(raw_expr)
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({_sql_arg(col_expr, df)})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        _column_arg(gb, df) for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "total_orders",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["unique_ffc27a21.unique_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_b89e1903.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_b89e1903" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_b89e1903.aggregated_data"])
if "aggregate_b89e1903" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_b89e1903", "counts": {"aggregated_data": int(ctx["aggregate_b89e1903.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_b89e1903_copy
template: aggregate
templateVersion: 2.0.0
name: Total number of orders_copy
position:
  x: 1145.1345818392265
  y: 132.82144203637063
description:
  text: Group data by customer ID and calculate the total sales sum.
  hash: 80d4cdb1
previewCodeHash: fecd263bc4611ee9
codegenHash: 39e07f1e5701539d
updatedAt: 1794916254199
config:
  group_bys:
    - expr: customer_id
      type: column
  aggregations:
    - columnExpr:
        expr: total_amount
        type: column
      fn: SUM
      alias: total_sales
checkpoint:
  mode: auto
input:
  - node: unique_ffc27a21
    input_port: data
    output_port: unique_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def _quoted_column(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _column_name(column_expr: Dict[str, Any], df) -> str:
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "column" and name != "*" and name in df.columns:
        return _quoted_column(name)
    return name

def _column_arg(column_expr: Dict[str, Any], df):
    name = column_expr.get("expr", "")
    if column_expr.get("type") == "expr":
        return F.expr(name)
    return F.col(_column_name(column_expr, df))

def _sql_arg(column_expr: Dict[str, Any], df) -> str:
    return _column_name(column_expr, df)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(_column_arg(col_expr, df))
        elif fn == "-" or fn == "_":
            col = _column_arg(col_expr, df)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({_sql_arg(col_expr, df)}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            col = F.concat_ws(sep, F.collect_list(_column_arg(col_expr, df)))
        elif fn == "COUNT_DISTINCT":
            arg = _column_arg(col_expr, df) if col_expr.get("type") == "column" else F.expr(raw_expr)
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({_sql_arg(col_expr, df)})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        _column_arg(gb, df) for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_id",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "total_amount",
                "type": "column"
            },
            "fn": "SUM",
            "alias": "total_sales",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["unique_ffc27a21.unique_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_b89e1903_copy.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_b89e1903_copy" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_b89e1903_copy.aggregated_data"])
if "aggregate_b89e1903_copy" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_b89e1903_copy", "counts": {"aggregated_data": int(ctx["aggregate_b89e1903_copy.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: sort_ca1988a3
template: sort
templateVersion: 1.0.0
name: Sort by sales (descending)
position:
  x: 1151.3885468344074
  y: 461.1243446722231
description:
  text: Sort data by total amount in descending order.
  hash: b3fe3289
previewCodeHash: ef1fc7845b33b17c
codegenHash: 8d8595d72d737d1d
updatedAt: 1794915626470
config:
  sort_expressions:
    - columnExpr:
        expr: total_amount
        type: expr
      sortBy: DESC
checkpoint:
  mode: auto
input:
  - node: unique_ffc27a21
    input_port: data
    output_port: unique_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "total_amount",
                "type": "expr"
            },
            "sortBy": "DESC"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["unique_ffc27a21.unique_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sort_ca1988a3.sorted_data"] = out["sorted_data"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["sort_ca1988a3.sorted_data"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["sort_ca1988a3.sorted_data"]
        try:
            ctx["sort_ca1988a3.sorted_data"] = ctx["sort_ca1988a3.sorted_data"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["sort_ca1988a3.sorted_data"] is not _lb_cp_original_0:
                try:
                    ctx["sort_ca1988a3.sorted_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["sort_ca1988a3.sorted_data"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "sort_ca1988a3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sort_ca1988a3.sorted_data"])
if "sort_ca1988a3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sort_ca1988a3", "counts": {"sorted_data": int(ctx["sort_ca1988a3.sorted_data"].count())}}}, raw=True)

In [0]:
"""
id: limit_806114b9
template: limit
templateVersion: 1.0.0
name: Highest sales transaction
position:
  x: 1440.9550826010507
  y: 462.76692999259217
description:
  text: Keep only the first N rows.
  hash: 1688a8eb
previewCodeHash: 64aab158f9a8a4d7
codegenHash: 74dc4d520a2a810c
updatedAt: 1794915626471
config:
  limit: "1"
checkpoint:
  mode: auto
input:
  - node: sort_ca1988a3
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    n = int(config.get("limit", 1000))

    return {"limited_data": df.limit(n)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "limit": "1"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sort_ca1988a3.sorted_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["limit_806114b9.limited_data"] = out["limited_data"]
if globals().get("ld_display_outputs", False) or "limit_806114b9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["limit_806114b9.limited_data"])
if "limit_806114b9" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "limit_806114b9", "counts": {"limited_data": int(ctx["limit_806114b9.limited_data"].count())}}}, raw=True)